# Fairness checks: food insecurity classifier

This notebook evaluates whether a food insecurity classifier performs differently across synthetic demographic and access groups. The protected or sensitive fields are excluded from model training, then used only for auditing.

Fairness metrics do not prove fairness. They are diagnostic checks that help identify where the model may need deeper review.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("synthetic_food_housing_insecurity_100k.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/synthetic_food_housing_insecurity_100k.csv")

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 1. Train an audit model without sensitive columns

The model excludes race/ethnicity, primary language, and immigrant household status from training. This does not remove all risk of disparity because other features can still act as proxies.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

TARGET = "food_insecure_label"
PROTECTED_COLUMNS = ["race_ethnicity", "primary_language", "immigrant_household"]
LEAKAGE_COLUMNS = [
    "household_id",
    "train_test_split",
    "food_insecure_label",
    "food_security_status",
    "overall_hardship_score",
    "policy_priority_segment",
    "meals_skipped_last_30d",
    "months_food_shortage_last_year",
    "pantry_use_last_30d",
    "housing_insecure_label",
] + PROTECTED_COLUMNS

features = [c for c in df.columns if c not in LEAKAGE_COLUMNS]
train_mask = df["train_test_split"].eq("train")

X_train = df.loc[train_mask, features]
y_train = df.loc[train_mask, TARGET]
X_test = df.loc[~train_mask, features]
y_test = df.loc[~train_mask, TARGET]

categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in features if c not in categorical_cols]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]), categorical_cols),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=500, solver="saga", class_weight="balanced", random_state=42)),
])
model.fit(X_train, y_train)

scores = model.predict_proba(X_test)[:, 1]
preds = (scores >= 0.50).astype(int)
print("ROC AUC:", round(roc_auc_score(y_test, scores), 3))

## 2. Define group fairness metrics

Common diagnostic metrics include selection rate, true positive rate, false positive rate, and precision. Selection rate means the share of a group predicted as high risk.

In [ ]:
audit_df = df.loc[
    ~train_mask,
    ["household_id", "race_ethnicity", "urbanicity", "disability_present", "children_present", "primary_language", TARGET],
].copy()
audit_df["score"] = scores
audit_df["prediction"] = preds


def group_metrics(data: pd.DataFrame, group_col: str, min_n: int = 200) -> pd.DataFrame:
    rows = []
    for group_value, g in data.groupby(group_col, dropna=False):
        if len(g) < min_n:
            continue
        y = g[TARGET].to_numpy()
        p = g["prediction"].to_numpy()
        tp = ((y == 1) & (p == 1)).sum()
        fp = ((y == 0) & (p == 1)).sum()
        tn = ((y == 0) & (p == 0)).sum()
        fn = ((y == 1) & (p == 0)).sum()
        rows.append({
            "group_column": group_col,
            "group": str(group_value),
            "n": len(g),
            "base_rate": y.mean(),
            "selection_rate": p.mean(),
            "true_positive_rate": tp / (tp + fn) if (tp + fn) else np.nan,
            "false_positive_rate": fp / (fp + tn) if (fp + tn) else np.nan,
            "precision": tp / (tp + fp) if (tp + fp) else np.nan,
            "avg_score": g["score"].mean(),
        })
    return pd.DataFrame(rows).sort_values(["group_column", "n"], ascending=[True, False])

fairness_tables = pd.concat([
    group_metrics(audit_df, "race_ethnicity"),
    group_metrics(audit_df, "urbanicity"),
    group_metrics(audit_df, "disability_present"),
    group_metrics(audit_df, "children_present"),
    group_metrics(audit_df, "primary_language"),
], ignore_index=True)

fairness_tables.round(3)

## 3. Summarize disparities

The table below shows the max-min gap and minimum-to-maximum ratio for each metric within each grouping column. These thresholds are not legal standards; they are screening indicators for review.

In [ ]:
def disparity_summary(metrics: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for group_col, g in metrics.groupby("group_column"):
        for metric in ["selection_rate", "true_positive_rate", "false_positive_rate", "precision"]:
            values = g[metric].dropna()
            if values.empty:
                continue
            rows.append({
                "group_column": group_col,
                "metric": metric,
                "min": values.min(),
                "max": values.max(),
                "max_minus_min": values.max() - values.min(),
                "min_to_max_ratio": values.min() / values.max() if values.max() else np.nan,
            })
    return pd.DataFrame(rows)

disparities = disparity_summary(fairness_tables)
disparities.round(3)

## 4. Visualize selected fairness metrics

This example plots true positive rate by race/ethnicity. Repeat the same pattern for any grouping column.

In [ ]:
import matplotlib.pyplot as plt

race_metrics = fairness_tables[fairness_tables["group_column"].eq("race_ethnicity")].sort_values("true_positive_rate")
plt.figure(figsize=(9, 5))
plt.barh(race_metrics["group"], race_metrics["true_positive_rate"])
plt.xlabel("True positive rate")
plt.title("True positive rate by race/ethnicity")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## 5. Export audit outputs

The fairness metrics can be reviewed with the model card or dashboard.

In [ ]:
fairness_tables.to_csv("food_insecurity_fairness_metrics.csv", index=False)
disparities.to_csv("food_insecurity_fairness_disparities.csv", index=False)
Path("food_insecurity_fairness_metrics.csv"), Path("food_insecurity_fairness_disparities.csv")